### 用戶提問前

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder 
from dotenv import load_dotenv
import anthropic
from config import FOLDER_PATH, EMBEDDING_MODEL, COLLECTION_NAME, CROSS_ENCODER
import re

In [ ]:
#把文章切成chunk
def split_into_chunks(doc_file: str) -> list[dict]:
    with open(doc_file, "r", encoding="utf-8") as file:
        content = file.read()
    #保留來源文件，for reference
    source = os.path.basename(doc_file)  
    chunks = [c.strip() for c in re.split(r"\n(?=### )", content) if c.strip()]
    return [
        {"id": f"{source}-{i}", "text": chunk, "source": source}
        for i, chunk in enumerate(chunks)
    ]
#回傳值為 a list of dicts 範例 [{id: "HR-01_工作規則.md-1", text:"### 第1條（目的）[...]。", source:"HR-01_工作規則.md"}, ...]

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
#把每個chunk轉換成向量
def embed_chunk(chunk:str) -> list[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()

In [ ]:
#建立向量資料庫
chromadb_client = chromadb.PersistentClient('./chroma_db')
collection = chromadb_client.get_or_create_collection(name=COLLECTION_NAME)

In [ ]:
#把得到的向量存入向量資料庫中

def save_embeddings(chunks: list[dict], embeddings:list[list[float]]) -> None:
    collection.upsert(
            ids=[c["id"] for c in chunks],
            documents=[c["text"] for c in chunks],
            embeddings=embeddings,
            metadatas=[{"source": c["source"]} for c in chunks],
        )
    print(f"已寫入 {len(chunks)} 段，資料庫共 {collection.count()} 段")
    

In [ ]:
#輸入query及top_k數值，回傳計算後top_k相近的chunk及其source
def retrieve(query:str, top_k:int) -> list[tuple[str, str]]:
    query_embed = embed_chunk(query)
    results = collection.query(
        query_embeddings=[query_embed],
        n_results=top_k
    )
    docs = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    return list(zip(docs, sources))

In [ ]:
#利用cross encoder重新排列打分數
#SentenceEncoder把每個chunk分別計算embedding，再比較向量之間的距離，且可存進ChromaDB中，速度較快
#CrossEncoder則把query和chunks都一起丟入模型中，較精準，但每次計算都是丟入（query+chunk)兩兩計算，因此速度很慢

reranker = CrossEncoder(CROSS_ENCODER)

#重新排列retrieve的chunks, 並得到top_k個結果，要給claude
def rerank(query: str, retrieved_chunks: list[tuple[str, str]], top_k: int) -> list[tuple[str, str]]:
    """retrieved_chunks 是 [(片段內容, 來源檔名), ...]"""
    if not retrieved_chunks:
        return []
    scores = reranker.predict([(query, text) for text, _ in retrieved_chunks])
    ranked = sorted(zip(retrieved_chunks, scores), key=lambda pair: pair[1], reverse=True)
    return [chunk for chunk, _ in ranked[:top_k]]
    

### 串接Claude (claude-sonnet-5-5)

In [ ]:
load_dotenv()
claude_client = anthropic.Anthropic()

In [ ]:
def generate(query: str, chunks: list[tuple[str, str]]) -> str:
    """chunks 是 [(片段內容, 來源檔名), ...]"""
    system_prompt = (
        "你是一個人資知識助手，請根據用戶的問題和提供的相關片段生成準確回答。"
        "請只根據相關片段回答，不要編造資訊。如果相關片段不足以回答，就回答「不知道」。"
        "每個片段開頭的【】內是它所屬的規章名稱，回答時請註明依據的規章名稱，"
        "例如「依據 HR-04_差勤管理辦法」。"
    )

    if not chunks:
        return "不知道。目前的規章中找不到相關內容。"

    context = "\n\n".join(f"【{source}】\n{text}" for text, source in chunks)
    user_prompt = f"用戶問題：{query}\n\n相關片段：\n{context}"


    response = claude_client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return response.content[0].text
    

In [ ]:
def answer(query: str, top_k_retrieved: int = 20, top_k_reranked: int = 5) -> str:
    retrieved_chunks = retrieve(query, top_k_retrieved)
    reranked_chunks = rerank(query=query, retrieved_chunks=retrieved_chunks, top_k=top_k_reranked)
    return generate(query, reranked_chunks)

In [ ]:
folder_path = FOLDER_PATH
list_of_chunk = []

#go through 整個 FOLDER_PATH (文件所在資料夾）
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if not file.endswith(".md"):
            continue
        file_path = os.path.join(root, file)
        list_of_chunk += split_into_chunks(file_path)

#取出list of chunk當中的文字部分
texts = [c["text"] for c in list_of_chunk]

#得到每個chunk的embeddings
embeddings = embedding_model.encode(texts, normalize_embeddings=True).tolist()

#把文件跟提取的向量放到向量數據庫中
save_embeddings(list_of_chunk, embeddings)

In [ ]:
query = "員工參加外部證照考試有補助報名費嗎？成功考取有沒有其他獎勵津貼？"
answer(query)